se importa la libreria y se lee el csv

In [ ]:
import pandas as pd

df = pd.read_csv('sales_data_sample.csv', 
                 encoding="latin-1")             

Crear un nuevo DataFrame con: ORDERNUMBER, PRODUCTLINE, COUNTRY, CUSTOMERNAME, QUANTITYORDERED, PRICEEACH, SALES, DEALSIZE.

In [ ]:
print(
    df[
        [
            "ORDERNUMBER",
            "PRODUCTLINE",
            "COUNTRY",
            "CUSTOMERNAME",
            "QUANTITYORDERED",
            "PRICEEACH",
            "SALES",
            "DEALSIZE",
        ]
    ].shape
)
df.head()

Filtrar únicamente los pedidos de la línea Vintage Cars.

In [ ]:

vintage = df[df["PRODUCTLINE"] == "Vintage Cars"]

print(f"Filas totales: {len(df)}")
print(f"Vintage Cars:  {len(vintage)}")
vintage.head()

Dentro de ese filtro, quedarse solo con los pedidos donde el país sea distinto de USA.

In [ ]:
notUsa = vintage[vintage["COUNTRY"] != "USA"]
print(f"solo Vintage Cars: {len(vintage)}")
print(f"Vintage cars not USA: {len(notUsa)}")
notUsa.head()

Ordenar ese resultado por SALES de mayor a menor y mostrar los primeros 10.

In [ ]:
# ascending=False -> de mayor a menor. Con True salian los 10 mas bajos.
ordenado = notUsa.sort_values(by="SALES", ascending=False)

# .head(n) devuelve las primeras n filas; sin argumento devuelve 5.
top10 = ordenado.head(10)

print(f"Total de pedidos: {len(ordenado)}")
print(f"Mostrando:        {len(top10)}")
top10

Responder con código: ¿qué país aparece con más frecuencia entre esos pedidos?

In [ ]:
# value_counts() cuenta cuantas veces se repite cada valor, ordenado de mayor a menor
conteo = df["COUNTRY"].value_counts()
print(conteo)
print()

# idxmax() -> la etiqueta con el conteo mas alto ; max() -> ese conteo
print(f"Pais mas frecuente: {conteo.idxmax()} ({conteo.max()} pedidos)")

Comparar ```df.loc[5]``` vs ```df.iloc[5]``` sobre el DataFrame ya ordenado, y explicar la diferencia en una frase.

In [ ]:
print("Indice de 'ordenado':", list(ordenado.index[:8]))
print()

# .loc busca por ETIQUETA del indice. Tras filtrar, la etiqueta 5 ya no existe.
try:
    print("ordenado.loc[5] :", ordenado.loc[5]["ORDERNUMBER"])
except KeyError:
    print("ordenado.loc[5] : KeyError -> la etiqueta 5 no esta en este DataFrame")

# .iloc busca por POSICION. La posicion 5 siempre existe si hay 6+ filas.
fila = ordenado.iloc[5]
print("ordenado.iloc[5]:", fila[["ORDERNUMBER", "COUNTRY", "SALES"]].tolist())
print()

# La etiqueta que vive en la posicion 5:
etiqueta = ordenado.index[5]
print(f"La posicion 5 tiene la etiqueta {etiqueta}")
print(f"ordenado.loc[{etiqueta}] == ordenado.iloc[5] ->",
      ordenado.loc[etiqueta].equals(ordenado.iloc[5]))

**Diferencia:** `.loc[5]` busca la fila cuya *etiqueta* de índice es 5, mientras que `.iloc[5]` busca la fila que ocupa la *posición* 5, sin importar cómo esté el índice.

Revisar el tipo de dato de ORDERDATE. ¿Llegó como fecha o como texto?

In [ ]:
print("dtype de ORDERDATE:", df["ORDERDATE"].dtype)

print("type del primer valor:", type(df["ORDERDATE"].iloc[0]))
print("valor de ejemplo:     ", repr(df["ORDERDATE"].iloc[0]))
print()


**Respuesta:** llegó como **texto**. El `dtype` es `object`, que es como pandas marca las columnas de strings, y cada valor es un `str` (`'2/24/2003 0:00'`). No es una fecha todavía.

Convertirla a fecha con `pd.to_datetime()`.

In [ ]:
print("ANTES: ", df["ORDERDATE"].dtype)

df["ORDERDATE"] = pd.to_datetime(df["ORDERDATE"], format="%m/%d/%Y %H:%M")

print("DESPUES:", df["ORDERDATE"].dtype)
print("Fechas que no se pudieron convertir:", df["ORDERDATE"].isna().sum())
print()

print("Pedido mas antiguo:", df["ORDERDATE"].min())
print("Pedido mas reciente:", df["ORDERDATE"].max())
print()
print(df["ORDERDATE"].head(3))

Convertir `COUNTRY` a tipo `category` y comparar `df.memory_usage(deep=True)` antes y despues.

In [ ]:
# deep=True obliga a medir el texto real de cada string.
# Sin deep=True solo cuenta los punteros (8 bytes por fila) y la mejora no se ve.
antes = df.memory_usage(deep=True)

df["COUNTRY"] = df["COUNTRY"].astype("category")

despues = df.memory_usage(deep=True)

comparacion = pd.DataFrame({"antes": antes, "despues": despues})
comparacion["diferencia"] = comparacion["despues"] - comparacion["antes"]
print(comparacion.loc[["COUNTRY"]])
print()

print(f"COUNTRY antes  : {antes['COUNTRY']:>9,} bytes")
print(f"COUNTRY despues: {despues['COUNTRY']:>9,} bytes")
print(f"reduccion      : {100 * (1 - despues['COUNTRY'] / antes['COUNTRY']):.1f}%")
print()
print(f"TOTAL antes    : {antes.sum():>9,} bytes")
print(f"TOTAL despues  : {despues.sum():>9,} bytes")
print(f"reduccion      : {100 * (1 - despues.sum() / antes.sum()):.1f}%")
print()

# Por que funciona: guarda 19 nombres una sola vez + un codigo entero por fila
print("dtype:      ", df["COUNTRY"].dtype)
print("categorias: ", len(df["COUNTRY"].cat.categories))

# Punto 12 — Reportes libres

## Reporte 1 — `PRICEEACH` está truncado en 100 y subestima el ingreso en $1.74M

**Qué se encontró:** la columna `PRICEEACH` tiene un tope artificial de 100. Ningún valor lo supera, y las 1304 filas que valen exactamente 100 (el 46% del dataset) son precisamente las que no cuadran con `SALES`.

**Cómo se comprobó:** se recalculó `QUANTITYORDERED * PRICEEACH` y se comparó con `SALES`. Las 1519 filas con `PRICEEACH < 100` cuadran al centavo (0 discrepancias). Las 1304 con `PRICEEACH == 100` fallan **todas**, y su precio implícito (`SALES / QUANTITYORDERED`) va de 100.01 a 252.87. El corte es artificial, no un dato real.

In [ ]:
calculado = df["QUANTITYORDERED"] * df["PRICEEACH"]
descuadre = (df["SALES"] - calculado).abs()

topados = df["PRICEEACH"] == 100

print(f'PRICEEACH maximo:            {df["PRICEEACH"].max()}')
print(f"Filas con PRICEEACH == 100:  {topados.sum()} de {len(df)}")
print(f"Filas que no cuadran:        {(descuadre > 0.01).sum()}")
print(f"  de ellas, topadas en 100:  {(topados & (descuadre > 0.01)).sum()}")
print(f"  con PRICEEACH < 100:       {(~topados & (descuadre > 0.01)).sum()}")
print()

# Precio real reconstruido desde SALES en las filas topadas
implicito = df.loc[topados, "SALES"] / df.loc[topados, "QUANTITYORDERED"]
print(f"Precio implicito: min {implicito.min():.2f} / max {implicito.max():.2f}")
print(f"Ingreso subestimado: ${descuadre.sum():,.2f}")

**Recomendación de negocio:** no usar `PRICEEACH` para márgenes, descuentos ni pricing — en casi la mitad de los pedidos está mal. `SALES` sí es confiable (viene del sistema origen), así que el precio unitario debe reconstruirse como `SALES / QUANTITYORDERED`. En paralelo, pedir a TI que corrija el truncamiento en la extracción: un análisis de precios hecho sobre esta columna concluiría que ningún producto supera los $100, lo cual es falso.

## Reporte 2 — Noviembre concentra el 21% del ingreso anual

**Qué se encontró:** la demanda es fuertemente estacional. Noviembre por sí solo aporta el 21.1% del ingreso, casi tres veces lo que aportaría un mes promedio (8.3%). El trimestre octubre–diciembre suma el 38.6%.

**Cómo se comprobó:** se agrupó `SALES` por mes de `ORDERDATE` (ya convertida a fecha en el paso 10) y se calculó el porcentaje sobre el total. El patrón se repite en 2003 y 2004, los dos años completos, así que no es el pico de un solo año.

In [ ]:
por_mes = df.groupby(df["ORDERDATE"].dt.month)["SALES"].sum()
pct = (100 * por_mes / por_mes.sum()).round(1)

print("% del ingreso anual por mes:")
print(pct)
print()
print(f"Noviembre:    {pct[11]}%  (un mes promedio seria 8.3%)")
print(f"Q4 (oct-dic): {pct[[10, 11, 12]].sum().round(1)}%")
print()

# Se repite el patron cada anio? 2003 y 2004 son los anios completos
completos = df[df["ORDERDATE"].dt.year.isin([2003, 2004])]
tabla = completos.pivot_table(index=completos["ORDERDATE"].dt.month,
                              columns=completos["ORDERDATE"].dt.year,
                              values="SALES", aggfunc="sum")
print("% del ingreso por mes, en cada anio completo:")
print((100 * tabla / tabla.sum()).round(1))

**Recomendación de negocio:** planificar inventario y personal contra el pico de noviembre, no contra el promedio anual. Un stock dimensionado con el promedio deja sin cubrir 1 de cada 5 dólares del año. Junio y julio (4.5% y 5.1%) son la ventana natural para mantenimiento, inventarios físicos y vacaciones del equipo.

## Reporte 3 — Dos clientes concentran el 15.6% del ingreso

**Qué se encontró:** de 92 clientes, *Euro Shopping Channel* aporta el 9.1% del ingreso total y *Mini Gifts Distributors* el 6.5%. Esos 2 clientes facturan tanto como los 29 más pequeños sumados ($1.57M contra $1.54M). El top 20 (22% de los clientes) concentra el 43.6%, y el cliente #1 vale 10.5 veces el cliente mediano.

**Cómo se comprobó:** se agrupó `SALES` por `CUSTOMERNAME`, se ordenó de mayor a menor y se calculó el porcentaje acumulado sobre el ingreso total. El corte de 29 se obtuvo sumando desde la cola hasta igualar a los 2 primeros.

In [ ]:
por_cliente = df.groupby("CUSTOMERNAME")["SALES"].sum().sort_values(ascending=False)
total = por_cliente.sum()

print(f"Clientes distintos: {len(por_cliente)}")
print()
print("Top 5 clientes:")
for nombre, monto in por_cliente.head(5).items():
    print(f"  {nombre:<32} ${monto:>12,.0f}   {100 * monto / total:>5.1f}%")
print()

for n in [1, 2, 5, 10, 20]:
    print(f"  top {n:>2} clientes -> {100 * por_cliente.head(n).sum() / total:5.1f}% del ingreso")
print()

# Los 2 mas grandes equivalen a cuantos de la cola?
dos = por_cliente.head(2).sum()
print(f"2 clientes mas grandes:  ${dos:,.0f}")
print(f"29 clientes mas chicos:  ${por_cliente.tail(29).sum():,.0f}")
print()
print(f"Cliente mediano: ${por_cliente.median():,.0f}")
print(f"El #1 vale {por_cliente.iloc[0] / por_cliente.median():.1f}x el mediano")

**Recomendación de negocio:** hay riesgo real de concentración — perder a *Euro Shopping Channel* borraría el 9% de la facturación de golpe. Se recomienda asignarles gestión de cuenta dedicada y contrato con renovación anticipada, y en paralelo abrir un plan de captación: la cola de 60 clientes pequeños es donde está el margen de crecimiento sin aumentar la dependencia.

## Reporte 4 — 2005 está incompleto: la “caída del 62%” es un espejismo

**Qué se encontró:** comparar los totales por año sugiere un desplome en 2005 ($4.72M en 2004 contra $1.79M en 2005, un -62%). Es falso: 2005 solo tiene datos hasta el 31 de mayo. Al comparar el mismo periodo en los tres años, el negocio **está creciendo**: enero–mayo pasó de $839K (2003) a $1.31M (2004) a $1.79M (2005).

**Cómo se comprobó:** se listaron los meses presentes en cada año — 2004 tiene los 12, 2005 solo del 1 al 5 — y se rehizo la comparación filtrando enero–mayo en los tres años. El signo de la conclusión se invierte por completo.

In [ ]:
anio = df["ORDERDATE"].dt.year

print("COMPARACION INGENUA (enganiosa):")
resumen = df.groupby(anio).agg(pedidos=("SALES", "size"),
                               ventas=("SALES", "sum"),
                               ultima_fecha=("ORDERDATE", "max"))
print(resumen)
print()

print("Meses con datos en cada anio:")
for a in sorted(anio.unique()):
    meses = sorted(df.loc[anio == a, "ORDERDATE"].dt.month.unique().tolist())
    print(f"  {a}: {len(meses)} meses -> {meses}")
print()

print("COMPARACION JUSTA (solo enero-mayo en los 3 anios):")
ene_may = df[df["ORDERDATE"].dt.month <= 5]
justa = ene_may.groupby(ene_may["ORDERDATE"].dt.year)["SALES"].sum()
print(justa.round(0))
print()
print("Crecimiento interanual ene-mayo (%):")
print((100 * justa.pct_change()).round(1).to_string())

**Recomendación de negocio:** ningún reporte de tendencia debe usar el total de 2005 como año cerrado. Se recomienda fijar como estándar la comparación *like-for-like* (mismo rango de meses) y marcar los periodos parciales en todo dashboard. El costo de no hacerlo es concreto: la lectura ingenua diría que el negocio se derrumbó un 62% y podría disparar recortes, cuando en realidad crece a doble dígito.